In [1]:
# Метод 2: Повторяющиеся n‑граммы высокого порядка
### Выделяются признаки: доля уникальных n‑грамм, доля самой частой n‑граммы, доля top‑K n‑грамм

!pip install -q scikit-learn pandas

In [2]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

df = pd.read_csv('/content/drive/MyDrive/LLMTrace_4class_20k.csv')

df['binary_label'] = df['label'].apply(lambda x: 0 if x in [0,3] else 1)
df['is_clean'] = df['label'].isin([0,1])

# Удалим выбросы
df['word_count'] = df['text'].apply(lambda x: len(x.split()))
def remove_outliers(df, genre_col='style', upper_quantile=0.95):
    cleaned = []
    for genre in df[genre_col].unique():
        human_subset = df[(df[genre_col]==genre) & (df['binary_label']==0)]
        if len(human_subset)>0:
            thresh = human_subset['word_count'].quantile(upper_quantile)
            human_filtered = human_subset[human_subset['word_count'] <= thresh]
            cleaned.append(human_filtered)
        ai_subset = df[(df[genre_col]==genre) & (df['binary_label']==1)]
        cleaned.append(ai_subset)
    return pd.concat(cleaned, ignore_index=True)
df = remove_outliers(df, upper_quantile=0.95)
df.drop(columns=['word_count'], inplace=True)

df['stratify'] = df['binary_label'].astype(str) + '_' + df['style']
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df['stratify']
)
train_full = df.loc[train_idx].copy()
test_full = df.loc[test_idx].copy()
train_full.drop(columns=['stratify'], inplace=True)
test_full.drop(columns=['stratify'], inplace=True)
print(f"Train: {len(train_full)}, Test: {len(test_full)}")

# 2. РЕАЛИЗАЦИЯ ДЕТЕКТОРА НА ОСНОВЕ ПОВТОРЯЮЩИХСЯ n-ГРАММ
class RepeatedNGramDetector:
    def __init__(self, n=4, top_k=1000, use_classifier=True):
        self.n = n
        self.top_k = top_k
        self.use_classifier = use_classifier
        self.clf = None

    def _extract_features(self, texts):
        features = []
        for text in texts:
            words = re.findall(r'\b\w+\b', text.lower())
            if len(words) < self.n:
                features.append([1.0, 1.0, 1.0])
                continue
            ngrams = [' '.join(words[i:i+self.n]) for i in range(len(words)-self.n+1)]
            total = len(ngrams)
            counter = Counter(ngrams)
            unique_ratio = len(counter) / total
            max_freq_ratio = max(counter.values()) / total
            top_counts = sum(cnt for _, cnt in counter.most_common(self.top_k))
            top_ratio = top_counts / total
            features.append([unique_ratio, max_freq_ratio, top_ratio])
        return np.array(features)

    def fit(self, X_text, y):
        features = self._extract_features(X_text)
        if self.use_classifier:
            self.clf = LogisticRegression(random_state=42, class_weight='balanced')
            self.clf.fit(features, y)
        return self

    def predict(self, X_text):
        features = self._extract_features(X_text)
        if self.use_classifier:
            return self.clf.predict(features)
        else:
            return (features[:, 0] < 0.5).astype(int)

# 3. ФУНКЦИЯ ОЦЕНКИ
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_AI': f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        'f1_Human': f1_score(y_test, y_pred, pos_label=0, zero_division=0),
        'precision_AI': precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        'recall_AI': recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    }



Train: 9730, Test: 2433


In [5]:

# 4. ПРОВЕДЕНИЕ ЭКСПЕРИМЕНТОВ
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

genres = df['style'].unique().tolist()
genre_column = 'style'
results = []

print("Эксперименты 1")
print("="*80)

train_clean = train_full[train_full['is_clean'] == True]
test_clean = test_full[test_full['is_clean'] == True]

# обучение на жанре -> тест на том же жанре / всех жанрах
for genre in genres:
    train_genre = train_clean[train_clean[genre_column] == genre]
    if len(train_genre) < 10:
        continue
    model = RepeatedNGramDetector(n=4, use_classifier=True)
    model.fit(train_genre['text'].values, train_genre['binary_label'].values)

    test_genre = test_clean[test_clean[genre_column] == genre]
    if len(test_genre) > 0:
        metrics = evaluate_model(model, test_genre['text'].values, test_genre['binary_label'].values)
        metrics.update({'experiment': '1A_clean_same_genre', 'genre_train': genre, 'genre_test': genre})
        results.append(metrics)

    if len(test_clean) > 0:
        metrics = evaluate_model(model, test_clean['text'].values, test_clean['binary_label'].values)
        metrics.update({'experiment': '1B_clean_cross_genre', 'genre_train': genre, 'genre_test': 'all'})
        results.append(metrics)

# обучение на всех чистых -> тест на каждом жанре
if len(train_clean) > 0:
    model_all = RepeatedNGramDetector(n=4, use_classifier=True)
    model_all.fit(train_clean['text'].values, train_clean['binary_label'].values)
    for genre in genres:
        test_genre = test_clean[test_clean[genre_column] == genre]
        if len(test_genre) > 0:
            metrics = evaluate_model(model_all, test_genre['text'].values, test_genre['binary_label'].values)
            metrics.update({'experiment': '1C_clean_all_train', 'genre_train': 'all', 'genre_test': genre})
            results.append(metrics)

print("Эксперименты 2")
print("="*80)

train_all = train_full
test_all = test_full

for genre in genres:
    train_genre = train_all[train_all[genre_column] == genre]
    if len(train_genre) < 10:
        continue
    model = RepeatedNGramDetector(n=4, use_classifier=True)
    model.fit(train_genre['text'].values, train_genre['binary_label'].values)

    test_genre = test_all[test_all[genre_column] == genre]
    if len(test_genre) > 0:
        metrics = evaluate_model(model, test_genre['text'].values, test_genre['binary_label'].values)
        metrics.update({'experiment': '2A_all_same_genre', 'genre_train': genre, 'genre_test': genre})
        results.append(metrics)

    if len(test_all) > 0:
        metrics = evaluate_model(model, test_all['text'].values, test_all['binary_label'].values)
        metrics.update({'experiment': '2B_all_cross_genre', 'genre_train': genre, 'genre_test': 'all'})
        results.append(metrics)

model_all = RepeatedNGramDetector(n=4, use_classifier=True)
model_all.fit(train_all['text'].values, train_all['binary_label'].values)
for genre in genres:
    test_genre = test_all[test_all[genre_column] == genre]
    if len(test_genre) > 0:
        metrics = evaluate_model(model_all, test_genre['text'].values, test_genre['binary_label'].values)
        metrics.update({'experiment': '2C_all_train', 'genre_train': 'all', 'genre_test': genre})
        results.append(metrics)

print("Эксперименты 3")
print("="*80)

test_unclean = test_full[test_full['is_clean'] == False]
if len(test_unclean) > 0:
    for genre in genres:
        train_genre = train_clean[train_clean[genre_column] == genre]
        if len(train_genre) < 10:
            continue
        model = RepeatedNGramDetector(n=4, use_classifier=True)
        model.fit(train_genre['text'].values, train_genre['binary_label'].values)

        test_unclean_genre = test_unclean[test_unclean[genre_column] == genre]
        if len(test_unclean_genre) > 0:
            metrics = evaluate_model(model, test_unclean_genre['text'].values, test_unclean_genre['binary_label'].values)
            metrics.update({'experiment': '3A_clean_same_genre_test_unclean', 'genre_train': genre, 'genre_test': f'unclean_{genre}'})
            results.append(metrics)

        if len(test_unclean) > 0:
            metrics = evaluate_model(model, test_unclean['text'].values, test_unclean['binary_label'].values)
            metrics.update({'experiment': '3B_clean_same_genre_test_all_unclean', 'genre_train': genre, 'genre_test': 'all_unclean'})
            results.append(metrics)

    if len(train_clean) > 0:
        model_all = RepeatedNGramDetector(n=4, use_classifier=True)
        model_all.fit(train_clean['text'].values, train_clean['binary_label'].values)
        for genre in genres:
            test_unclean_genre = test_unclean[test_unclean[genre_column] == genre]
            if len(test_unclean_genre) > 0:
                metrics = evaluate_model(model_all, test_unclean_genre['text'].values, test_unclean_genre['binary_label'].values)
                metrics.update({'experiment': '3C_clean_all_train_test_unclean_per_genre', 'genre_train': 'all_clean', 'genre_test': f'unclean_{genre}'})
                results.append(metrics)

# 5. ВЫВОД РЕЗУЛЬТАТОВ
df_results = pd.DataFrame(results)
df_results['method'] = 'RepeatedNGram'
print("\n" + "="*80)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ (Repeated n-grams)")
print("="*80)
print(df_results.to_string(index=False))



Эксперименты 1: только чистые тексты (is_clean == True)
Эксперименты 2: все данные (с перефразированными)
Эксперименты 3: устойчивость к атакам (обучение на чистых, тест на нечистых)

ИТОГОВЫЕ РЕЗУЛЬТАТЫ (Repeated n-grams)
 accuracy    f1_AI  f1_Human  precision_AI  recall_AI                                experiment genre_train          genre_test        method
 0.536101 0.568067  0.499025      0.473389   0.710084                       1A_clean_same_genre  literature          literature RepeatedNGram
 0.549550 0.643947  0.387040      0.541952   0.793233                      1B_clean_cross_genre  literature                 all RepeatedNGram
 0.607080 0.693370  0.453202      0.604819   0.812298                       1A_clean_same_genre publicistic         publicistic RepeatedNGram
 0.531532 0.600877  0.433022      0.534113   0.686717                      1B_clean_cross_genre publicistic                 all RepeatedNGram
 0.537931 0.424069  0.614203      0.755102   0.294821              

In [6]:
df_results.to_csv('/content/drive/MyDrive/repeated_ngram_results.csv', index=False)


Результаты сохранены в 'repeated_ngram_results.csv'
